In [1]:
from dataset.documents import load_vietnamese_legal_documents,chunk_documents,load_vietnamese_legal_datasets
from dataset.vectorstore import VectorStoreDB, VectorStoreType
from rag.llm.embeddings import embedding_factory, EmbeddingProvider
from dataset.sql import SQLiteDatabase, VietnamLawModel

from settings.settings import (
    EMBEDDING_MODEL,
    SENTENCE_TRANFORMER_MODEL,
    LLM_MODEL,
    PERSIST_DIR,
    CHUNK_SIZE,
    CHUNK_OVERLAPPED,
 )

f:\AI Project\corporation_law_vietnam_rag\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
docs = load_vietnamese_legal_documents(start=15000,limit=5000) #15000
datasets = load_vietnamese_legal_datasets(start=15000,limit=5000) #15000
chunked_doc = chunk_documents(docs, chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAPPED)
embedder = embedding_factory(EmbeddingProvider.HUGGINGFACE, EMBEDDING_MODEL)
db = VectorStoreDB(
    type=VectorStoreType.QDRANT,
    collection_name="vietnamese_legal_docs",
    embedder=embedder
)
sql = SQLiteDatabase(name="vietnam_laws.db", path = "../db/sql", model=VietnamLawModel)

In [ ]:
docs[0]

In [ ]:
sql.add_many(datasets)

In [5]:
print(f"data = {sql.get_at_index(0)}")

data = id=2153 document_number='105/2005/QĐ-UBND' content='ỦY BAN NHÂN DÂN THÀNH PHỐ HỒ CHÍ MINH ****** | CỘNG HOÀ XÃ HỘI CHỦ NGHĨA VIỆT NAM Độc lập - Tự do - Hạnh phúc ********\nSố: 105/2005/QĐ-UBND | TP. Hồ Chí Minhi, ngày 16 tháng 06 năm 2005\n\nQUYẾT ĐỊNH\n\nVỀ PHÊ DUYỆT ĐIỀU LỆ TỔ CHỨC HOẠT ĐỘNG HỘI NHA CÔNG THÀNH PHỐ HỒ CHÍ MINH\n\nỦY BAN NHÂN DÂN THÀNH PHỐ HỒ CHÍ MINH\n\nCăn cứ Luật Tổ chức Hội đồng nhân dân và Ủy ban nhân dân ngày 26 tháng 11 năm 2003 ; Căn cứ Sắc lệnh số 102/SL/L004 ngày 20 tháng 5 năm 1957 ban hành Luật quy định về quyền lập Hội ; Căn cứ Nghị định số 88/2003/NĐ-CP ngày 30 tháng 7 năm 2003 của Chính phủ quy định về tổ chức, hoạt động và quản lý Hội ; Theo biên bản Đại hội đại biểu Hội Nha công thành phố Hồ Chí Minh ngày 17 tháng 3 năm 2005 ; Xét đề nghị của Chủ tịch Hội Nha công thành phố Hồ Chí Minh tại Văn bản số 04/NC ngày 04 tháng 4 năm 2005 và của Giám đốc Sở Nội vụ tại Tờ trình số 295/TTr-SNV ngày 06 tháng 6 năm 2005 ;\n\nQUYẾT ĐỊNH:\n\nĐiều 1. Nay phê d

In [ ]:
db.build()

In [ ]:
db.add(
    documents=chunked_doc,
    batch_size=16,
    wait=False,
    timeout=120,
)